In [ ]:
# apktool.bat -version
# apktool.bat d D:\com.jvstudios.gpstracker-254.apk

Use apktool to extract APKs from the specified directory into an APK output directory; see `dicompile_apks.ipynb`.

In [ ]:
# !D:/softwall_install/apktool/apktool.bat --version 2.12.1
# D:/softwall_install/apktool/apktool.bat d D:\TTU\research\location_privacy_compliance\project\open_apk\raw_apks\com.jvstudios.gpstracker-258.apk -f -o D:\TTU
# -s means skip sources and do not decompile code, improving speed by more than 10x

A Python script that can be applied directly to a directory extracted with 7-Zip to perform the complete workflow:

Global scan: extract `http(s)://...` URLs from every file in the extracted directory, including both text and binary files, and record the source file for each hit.

Classify by source: dex / res / assets / lib / manifest / other.

Then locate usage sites (class/method):

Preferred: if an `smali*` directory extracted by apktool is available, locate the exact `.method` in the smali files.

Otherwise: if jadx is installed locally (jadx-cli is recommended), invoke it automatically to parse the decompiled code and locate the surrounding class/method.

Output: generate both JSON (easiest to read) and CSV (convenient for Excel/statistics).

I recommend generating both: use JSON to inspect the structure and evidence chain, and CSV to filter, sort, aggregate, and create tables.

In [5]:
import argparse
import csv
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
from collections import defaultdict
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import re, json, csv, hashlib
from collections import defaultdict
from datetime import datetime
from urllib.parse import urlparse
import re
import pandas as pd
from collections import Counter

In [ ]:
# URL_BYTES_RE = re.compile(
#     br"https?://[A-Za-z0-9\-\._~:/\?#\[\]@!\$&'\(\)\*\+,;=%]+"
# )

# Start with a permissive scan, but do not cross quotes, whitespace, angle brackets, or backslashes.
URL_BYTES_RE = re.compile(
    rb'https?://[^\s"\'<>{}\\|`]+',
    re.IGNORECASE
)

# Split a concatenated string into multiple http/https starting points.
HTTP_SPLIT_RE = re.compile(r'https?://', re.IGNORECASE)

# Common suffixes for webpage/resource URLs; extend as needed.
# LIKELY_URL_END_RE = re.compile(
#     r'''(?ix)
#     ^
#     (
#         https?://.*?
#         (?:
#             \.html? |
#             \.xhtml |
#             \.php |
#             \.asp(?:x)? |
#             \.jsp |
#             \.json |
#             \.xml |
#             \.js |
#             \.css |
#             \.svg |
#             \.png |
#             \.jpg |
#             \.jpeg |
#             \.webp |
#             \.gif |
#             \.ico |
#             / |
#             \? |
#             \#
#         )
#     )
#     '''
# )
LIKELY_URL_END_RE = re.compile(
    r'''(?ix)
    ^
    (
        https?://.*?
        (?:
            \.html? |
            ...
            / |
            \? |
            \#
        )
    )
    '''
)

# ---- precise smali mapping ----
SMALI_CLASS_RE = re.compile(r"^\.class\b.*\s(L.+;)\s*$")
SMALI_METHOD_RE = re.compile(r"^\.method\b(.*)$")
SMALI_END_METHOD_RE = re.compile(r"^\.end method\b")
SMALI_CONST_STRING_RE = re.compile(r'^\s*const-string(?:/jumbo)?\s+[^,]+,\s+"(.*)"\s*$')

# ---- 1) Rule library: third-party domains and package-name keywords (extend as needed) ----
THIRD_PARTY_DOMAIN_HINTS = [
    "google.com", "googleapis.com", "gstatic.com",
    "firebase", "firebasestorage.googleapis.com",
    "mapbox.com", "api.mapbox.com",
    "doubleclick.net", "googlesyndication.com",
    "facebook.com", "fbcdn.net",
    "appsflyer.com", "adjust.com",
    "branch.io", "segment.com", "amplitude.com", "mixpanel.com",
    "crashlytics", "sentry.io",
    "app-measurement.com",
]

# These paths in smali/jadx files usually indicate third-party SDK code.
THIRD_PARTY_CODE_HINTS = [
    "/com/google/",
    "/com/firebase/",
    "/com/mapbox/",
    "/com/facebook/",
    "/com/appsflyer/",
    "/com/adjust/",
    "/io/sentry/",
    "/com/segment/",
    "/com/amplitude/",
    "/com/mixpanel/",
    "/okhttp3/", "/retrofit2/",
    "/com/airbnb/",  # Adjust as needed.
]

# ---- 3) Types that are not webpages intended for users (common resource suffixes) ----
NON_PAGE_EXT = (".js",".css",".png",".jpg",".jpeg",".webp",".gif",".svg",".ico",
                ".json",".mp3",".mp4",".wav",".zip",".apk",".dex",".so")

# ---- 2) API / telemetry endpoint patterns ----
API_PATH_PATTERNS = [
    r"/api(/|$)", r"/v\d+(/|$)", r"/graphql(/|$)",
    r"/track(/|$)", r"/collect(/|$)", r"/event(s)?(/|$)",
    r"/log(s)?(/|$)", r"/analytics(/|$)", r"/metrics(/|$)",
    r"/measurement(/|$)", r"/telemetry(/|$)"
]
API_PATH_RE = re.compile("|".join(API_PATH_PATTERNS), re.IGNORECASE)

In [ ]:
# URL scanning + classification + precise smali mapping (class/method/line)
def normalize_url(u: str) -> str:
    return u.rstrip('"\')]>},.;:')

def is_probably_valid_url(u: str) -> bool:
    try:
        p = urlparse(u)
        return p.scheme in ("http", "https") and bool(p.netloc)
    except Exception:
        return False

def trim_concatenated_tail(u: str) -> str:
    """
    First remove a second concatenated http/https URL,
    then remove unrelated trailing characters.
    """
    u = normalize_url(u)

    # 1) Check whether another http/https URL is concatenated later.
    m2 = re.search(r'(?i)https?://', u[8:])   # Skip the initial URL prefix.
    if m2:
        u = u[:8 + m2.start()]

    u = normalize_url(u)

    # 2) If the tail resembles a camel-case function or variable name, trim it based on a resource suffix.
    m = re.match(
        r'(?i)^(.+?\.(?:html?|xhtml|php|aspx?|jsp|json|xml|js|css|svg|png|jpg|jpeg|webp|gif|ico))(?:[A-Z_].*)?$',
        u
    )
    if m:
        candidate = m.group(1)
        if is_probably_valid_url(candidate):
            return candidate

    return u

def classify_source(rel_path: str) -> str:
    p = rel_path.replace("\\", "/")
    name = p.lower()
    if name.endswith(".dex"):
        return "dex"
    if name.endswith(".so"):
        return "lib_native"
    if p.startswith("assets/"):
        return "assets"
    if p.startswith("res/"):
        return "res"
    if name.endswith("androidmanifest.xml"):
        return "manifest"
    if name.endswith(".arsc"):
        return "resources_arsc"
    if p.startswith("smali") and name.endswith(".smali"):
        return "smali"
    if p.startswith("unknown/"):
        return "unknown"
    return "other"

def file_sha1_1mb(path: Path) -> str:
    h = hashlib.sha1()
    with path.open("rb") as f:
        h.update(f.read(1024 * 1024))
    return h.hexdigest()

def looks_like_real_page_or_asset_1(u: str) -> bool:
    try:
        p = urlparse(u)
        path = (p.path or "").lower()
        return (
            "privacy" in u.lower()
            or path.endswith((".html", ".htm", ".php", ".aspx", ".jsp", ".json", ".xml",
                              ".png", ".jpg", ".jpeg", ".svg", ".webp", ".css", ".js"))
            or "/" in path
        )
    except Exception:
        return False
    
def looks_like_real_page_or_asset(u: str) -> bool:
    try:
        p = urlparse(u)
        host = (p.netloc or "").lower()
        path = (p.path or "").lower()

        if not host:
            return False

        return (
            "privacy" in u.lower()
            or path.endswith((
                ".html", ".htm", ".php", ".aspx", ".jsp", ".json", ".xml",
                ".png", ".jpg", ".jpeg", ".svg", ".webp", ".css", ".js"
            ))
            or len(path) > 1
        )
    except Exception:
        return False

HTTP_START_RE = re.compile(rb'https?://', re.IGNORECASE)

def extract_urls_from_bytes(data: bytes): #, max_window: int = 2048
    for m in URL_BYTES_RE.finditer(data):
        # start = m.start()
        # chunk = data[start:start + max_window]
        # try:
        #     raw = chunk.decode("utf-8", errors="ignore")
        # except Exception:
        #     raw = chunk.decode("latin1", errors="ignore")
        # # Truncate at an obvious delimiter.
        # raw = re.split(r'[\s"\'<>{}\\|`]', raw, maxsplit=1)[0]
        # raw = trim_concatenated_tail(raw)
        # if not raw:
        #     continue
        # if not is_probably_valid_url(raw):
        #     continue
        # if not looks_like_real_page_or_asset(raw):
        #     continue

        # yield raw, start


        try:
            raw = m.group(0).decode("utf-8", errors="ignore")
        except Exception:
            raw = m.group(0).decode("latin1", errors="ignore")

        raw = normalize_url(raw)
        if not raw:
            continue

        # Split a hit into multiple URLs if multiple http/https prefixes occur.
        starts = [x.start() for x in HTTP_SPLIT_RE.finditer(raw)]
        if not starts:
            continue

        

        starts.append(len(raw))

        for i in range(len(starts) - 1):
            piece = raw[starts[i]:starts[i + 1]]

            # For safety, keep only the portion beginning with http(s)://.
            m2 = HTTP_SPLIT_RE.search(piece)
            if not m2:
                continue
            piece = piece[m2.start():]

            piece = trim_concatenated_tail(piece)

            if not is_probably_valid_url(piece):
                continue

            if not looks_like_real_page_or_asset(piece):
                continue

            if not piece:
                continue
            

            yield piece, m.start() + starts[i]

def scan_all_files(root: Path, max_mb: int = 50):
    max_bytes = max_mb * 1024 * 1024
    recs = []
    for p in root.rglob("*"):
        if not p.is_file():
            continue
        try:
            size = p.stat().st_size
            if size > max_bytes:
                continue
            data = p.read_bytes()
        except Exception:
            continue

        hits = list(extract_urls_from_bytes(data))
        if not hits:
            continue

        rel = str(p.relative_to(root)).replace("\\", "/")
        src = classify_source(rel)
        sha = file_sha1_1mb(p)

        for url, off in hits:
            recs.append({
                "url": url,
                "file": rel,
                "source": src,
                "offset": off,
                "size_bytes": size,
                "file_sha1_1mb": sha,
            })
    # Deduplicate by (url, file, offset).
    seen = set()
    out = []
    for r in recs:
        k = (r["url"], r["file"], r["offset"])
        if k in seen:
            continue
        seen.add(k)
        out.append(r)
    return out

def map_urls_in_smali(root: Path, target_urls: set):
    usage = defaultdict(list)
    smali_dirs = [p for p in root.iterdir() if p.is_dir() and p.name.startswith("smali")]
    for sd in smali_dirs:
        for smali_file in sd.rglob("*.smali"):
            try:
                lines = smali_file.read_text(encoding="utf-8", errors="ignore").splitlines()
            except Exception:
                continue

            cur_class = None
            cur_method = None
            in_method = False

            for idx, line in enumerate(lines, start=1):
                m = SMALI_CLASS_RE.match(line)
                if m:
                    cur_class = m.group(1)

                m = SMALI_METHOD_RE.match(line)
                if m:
                    cur_method = m.group(1).strip()
                    in_method = True

                if SMALI_END_METHOD_RE.match(line):
                    in_method = False
                    cur_method = None

                m = SMALI_CONST_STRING_RE.match(line)
                if m:
                    s = m.group(1)
                    if "http://" in s or "https://" in s:
                        for url, _ in extract_urls_from_bytes(s.encode("utf-8", errors="ignore")):
                            if url in target_urls:
                                usage[url].append({
                                    "engine": "smali",
                                    "smali_file": str(smali_file.relative_to(root)).replace("\\", "/"),
                                    "class": cur_class,
                                    "method": cur_method if in_method else None,
                                    "line": idx,
                                    "snippet": line.strip(),
                                })
    return usage

# Quickly filter likely privacy-policy URLs.
def urls_contain_keyword(urls_sorted):
    privacy_keywords = ("privacy", "privacy-policy", "privacypolicy")

    candidates = []
    for u in urls_sorted:
        lu = u.lower()
        if any(k in lu for k in privacy_keywords):
            candidates.append(u)

    print("Privacy-like candidates:", len(candidates))
    for u in candidates[:50]:
        print("-", u)
    return candidates


# ---- 2) API / telemetry endpoint patterns ----

# ---- 3) Non-user-facing webpage types (common resource suffixes) ----

def host_of(url: str) -> str:
    try:
        return (urlparse(url).netloc or "").lower()
    except Exception:
        return ""

def path_of(url: str) -> str:
    try:
        return (urlparse(url).path or "").lower()
    except Exception:
        return ""

def looks_like_api_or_tracking(url: str) -> bool:
    p = path_of(url)
    if API_PATH_RE.search(p):
        return True
    # Common telemetry parameters in the query string also count (optional).
    q = (urlparse(url).query or "").lower()
    if any(k in q for k in ["utm_", "gclid", "fbclid", "event", "click", "adid"]):
        return True
    return False
    

def looks_like_third_party_domain(url: str) -> bool:
    h = host_of(url)
    return any(h.endswith(d) or d in h for d in THIRD_PARTY_DOMAIN_HINTS)

def location_looks_like_third_ad(url: str, file_path) -> bool:
    # 1. Define the third-party/advertising keyword blacklist.
    AD_KEYWORDS = ['ad-', 'ads', 'tracker', 'analytics', 'doubleclick', 'google-ad']
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    for item in data.get('urls', []):
        if item.get('url') == url:
            # After finding a match, iterate through all hits for that URL.
            for hit in item.get('hits', []):
                file_path = hit.get('file').lower()
                source_type = hit.get('source').lower()

                # 2. Check whether the path contains a blacklist keyword.
                is_third_party = any(kw in file_path.lower() for kw in AD_KEYWORDS)
                if is_third_party:
                    return True
    return False            


def has_non_page_extension(url: str) -> bool:
    p = path_of(url)
    return any(p.endswith(ext) for ext in NON_PAGE_EXT)

def usage_is_third_party(usage_map,url: str) -> bool:
    """
    Classify a URL as third-party if its usage (smali file path) clearly falls under a third-party package path.
    This is more reliable than the domain because an app's policy may be hosted on a third-party domain.
    """
    for u in usage_map.get(url, []):
        f = (u.get("smali_file") or "").lower()
        if any(h in f for h in THIRD_PARTY_CODE_HINTS):
            return True
    return False

def app_package_hint_from_root(root: Path) -> str:
    """
    Roughly infer the app package name by finding one of the most frequent top-level packages under smali*.
    The package can also be parsed directly from package= in AndroidManifest.xml; this keeps the implementation simple.
    """
    # For a more precise result, parse package="" in out_jv/AndroidManifest.xml.
    return ""

# APP_PKG_HINT = app_package_hint_from_root(ROOT)

def usage_is_app_code(usage_map, app_prefixes, url: str) -> bool:
    """
    Treat usage as app code when the smali_file appears under the app's own package path (for example, /com/jvstudios/).
    You can manually change this to the app's package prefix, such as "/com/jvstudios/".
    """
    # Manually specifying the package prefix is most reliable (example: jvstudios).
    # app_prefixes = ["com/locator/gpstracker/phone"]  # <- Adjust for each app, such as /com/locator/ or /com/jvstudios/gpstracker/.
    for u in usage_map.get(url, []):
        f = (u.get("smali_file") or "").lower()
        if any(pref in f for pref in app_prefixes):
            return True
    return False

In [38]:
# path = "two_examples/dicompile"
# folders = [item for item in os.listdir(path) if os.path.isdir(os.path.join(path, item))]
# folders

# new_folders = [[f"/{f.split('-')[0].replace('.', '/')}/"] for f in folders]
# new_folders

# # [['/com/jvstudios/gpstracker/'],
# #  ['/com/jvstudios/gpstracker/'],
# #  ['/com/locator/gpstracker/phone/'],
# #  ['/com/locator/gpstracker/phone/']]

In [ ]:
### AA1_first_batch
# [
# "1122apk\dicompile1122apk\AA1_first_batch\AA1_first_328_batch"
# 1122apk\dicompile1122apk\AA1_first_batch\AA2_second_100_batch
# 1122apk\dicompile1122apk\AA1_first_batch\AA3_third_100_batch
# 1122apk\dicompile1122apk\AA1_first_batch\AA4_forth_100_batch
# 1122apk\dicompile1122apk\AA1_first_batch\AA5_fifth_100_batch
# 1122apk\dicompile1122apk\AA1_first_batch\AA6_sisth_74_batch
# ]

# [
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA1_first_328_batch
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA2_second_100_batch
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA3_third_100_batch
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA4_forth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA5_fifth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA6_sisth_100_batch
# ]

### AA2_second_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA1_first_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA2_second_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA3_third_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA4_forth_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA5_fifth_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA6_sisth_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA7_seventh_100_batch

# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA1_first_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA2_second_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA3_third_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA4_forth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA5_fifth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA6_sisth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA7_seventh_100_batch

### AA3_third_time
# 1122apk\dicompile1122apk\AA3_third_batch\AA1_first_100_batch
# 1122apk\dicompile1122apk\AA3_third_batch\AA2_second_100_batch
# 1122apk\dicompile1122apk\AA3_third_batch\AA3_third_100_batch
# 1122apk\dicompile1122apk\AA3_third_batch\AA4_forth_100_batch
# 1122apk\dicompile1122apk\AA3_third_batch\AA5_fifth_100_batch
# 1122apk\dicompile1122apk\AA3_third_batch\AA6_sisth_100_batch
# 1122apk\dicompile1122apk\AA3_third_batch\AA7_seventh_100_batch

# 1122apk\1122apk_privacy_policy_url\AA3_third_batch\AA1_first_100_batch
# 1122apk\1122apk_privacy_policy_url\AA3_third_batch\AA2_second_100_batch
# 1122apk\1122apk_privacy_policy_url\AA3_third_batch\AA3_third_100_batch
# 1122apk\1122apk_privacy_policy_url\AA3_third_batch\AA4_forth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA3_third_batch\AA5_fifth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA3_third_batch\AA6_sisth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA3_third_batch\AA7_seventh_100_batch


### AA4_forth_time
# 1122apk\dicompile1122apk\AA4_fourth_batch

# 1122apk\1122apk_privacy_policy_url\AA4_fourth_batch

In [ ]:
path = "1122apk\dicompile1122apk\AA4_fourth_batch"  # "1122apk/20_examples/test1apk" # "1122apk/dicompile1122apk" # "two_examples/dicompile" # "1122apk\dicompile1122apk"


folders = [item for item in os.listdir(path) if os.path.isdir(os.path.join(path, item))]
for folder in folders:
    # cur_path = Path(rf"{path}/{folder}")
    # print(cur_path)
    # Change this to your apktool output directory.
    ROOT = Path(rf"{path}/{folder}")  # 例如 r"D:\apktool_out\out_jv" "apk/com.jvstudios.gpstracker-258" "apk/com.locator.gpstracker.phone-153-apktool"
    print(ROOT)
    app_name = folder
    prefix_path = folder.split('-')[0].replace('.', '/')
    app_prefixes = [f"/{prefix_path}/"]
    print(app_prefixes)
    OUT_DIR = Path(r"1122apk\1122apk_privacy_policy_url\AA4_fourth_batch") 
    # [] 
    OUT_DIR.mkdir(parents=True, exist_ok=True)



    JSON_PATH = OUT_DIR / f"{app_name}_urls.json"
    CSV_PATH  = OUT_DIR / f"{app_name}_urls.csv"

    Original_Path = OUT_DIR / f"{app_name}_original_urls.csv"

    # print("ROOT:", ROOT.resolve())
    # print("OUT :", OUT_DIR.resolve())

    # ===== URL scanning + classification + precise Smali localization（class/method/line）=====
    records = scan_all_files(ROOT, max_mb=50)
    target_urls = set(r["url"] for r in records)
    usage_map = map_urls_in_smali(ROOT, target_urls)

    print("Total hits:", len(records))
    print("Unique URLs:", len(target_urls))
    print("URLs with smali usage:", sum(1 for u in target_urls if u in usage_map))
    # for url in target_urls:
    #     print(url)
    # JSON_PATH.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")

    # ==== Filter URLs containing keywords, categorize them by source, and output as JSON or CSV. ====
    by_source = Counter(r["source"] for r in records)
    print("By source:", by_source)
    urls_sorted = sorted(target_urls)
    candidates = urls_contain_keyword(urls_sorted)
    payload = {
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "summary": {
            "total_hits": len(records),
            "unique_urls": len(candidates),
            "by_source": dict(by_source),
        },
        "urls": [
            {
                "url": u,
                "hits": [r for r in records if r["url"] == u],
                "usage": usage_map.get(u, []),
            }
            for u in candidates
        ],
    }

    # ===== output JSON / CSV ====
    JSON_PATH.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Wrote JSON:", JSON_PATH)

    # CSV (flat)
    fields = [
        "url","source","file","offset","size_bytes","file_sha1_1mb",
        "usage_engine","usage_smali_file","usage_class","usage_method","usage_line"
    ]
    with CSV_PATH.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        for r in records:
            usages = usage_map.get(r["url"], [])
            if not usages:
                w.writerow({**r,
                            "usage_engine":"",
                            "usage_smali_file":"",
                            "usage_class":"",
                            "usage_method":"",
                            "usage_line":""})
            else:
                for u in usages:
                    w.writerow({**r,
                                "usage_engine": u.get("engine",""),
                                "usage_smali_file": u.get("smali_file",""),
                                "usage_class": u.get("class",""),
                                "usage_method": u.get("method",""),
                                "usage_line": u.get("line","")})

    print("Wrote CSV :", CSV_PATH)



    # ---- Exclude and provide reasons for the exclusion. ----
    kept = []
    dropped = []

    for url in candidates:
        reasons_drop = []
        reasons_keep = []

        if has_non_page_extension(url):
            reasons_drop.append("non-page extension (asset/media/script)")

        # API/Prioritize kicking off users with event tracking data.
        if looks_like_api_or_tracking(url):
            reasons_drop.append("looks like API/tracking endpoint")

        # If the usage is clearly found in a third-party SDK package, kick
        if usage_is_third_party(usage_map, url):
            reasons_drop.append("used inside third-party SDK package (smali path)")

        # If the domain resembles a third-party service, it isn't immediately rejected (since policies might also be hosted by third parties), but it is treated as a weak signal.
        if looks_like_third_party_domain(url):
            reasons_drop.append("third-party hosting domain (weak signal)")

        # If the location of the URL looks like a third-party or ad-related path, kick it out.
        if location_looks_like_third_ad(url, JSON_PATH):
            reasons_drop.append("location of url looks like third or ad")
    

        # Conversely, if it is referenced within the app's own code path, it serves as a strong retention signal.
        if usage_is_app_code(usage_map, app_prefixes, url):
            reasons_keep.append("referenced from app package code (smali path)")
            # If there is a strong retention signal, exclude "weak signals" (such as third-party domains) from the drop list.
            reasons_drop = [r for r in reasons_drop if "third-party hosting domain" not in r]

        # Final verdict: Discard if there is any compelling reason to drop the item and no strong grounds for retention.
        hard_drop = any(r in reasons_drop for r in [
            "non-page extension (asset/media/script)",
            "looks like API/tracking endpoint",
            "used inside third-party SDK package (smali path)",
            "location of url looks like third or ad",
        ])

        if hard_drop and not reasons_keep:
            dropped.append({"url": url, "drop_reasons": reasons_drop})
        else:
            kept.append({"url": url, "keep_reasons": reasons_keep, "warnings": reasons_drop})

    print("Candidates:", len(candidates))
    print("Kept      :", len(kept))
    print("Dropped   :", len(dropped))

    print("\n--- KEPT (top 30) ---")
    for x in kept[:30]:
        print("-", x["url"])
        if x["keep_reasons"]:
            print("   keep:", "; ".join(x["keep_reasons"]))
        if x["warnings"]:
            print("   warn:", "; ".join(x["warnings"]))

    print("\n--- DROPPED (top 30) ---")
    for x in dropped[:30]:
        print("-", x["url"])
        print("   drop:", "; ".join(x["drop_reasons"]))

    df_kept = pd.DataFrame(kept)
    df_dropped = pd.DataFrame(dropped)

    (df_kept).to_csv(OUT_DIR / f"{app_name}_privacy_candidates_kept.csv", index=False, encoding="utf-8")
    (df_dropped).to_csv(OUT_DIR / f"{app_name}_privacy_candidates_dropped.csv", index=False, encoding="utf-8")

    (OUT_DIR / f"{app_name}_privacy_candidates_kept.json").write_text(json.dumps(kept, indent=2, ensure_ascii=False), encoding="utf-8")
    (OUT_DIR / f"{app_name}_privacy_candidates_dropped.json").write_text(json.dumps(dropped, indent=2, ensure_ascii=False), encoding="utf-8")

    print("Wrote:", OUT_DIR / f"{app_name}_privacy_candidates_kept.csv")
    print("Wrote:", OUT_DIR / f"{app_name}_privacy_candidates_dropped.csv")

1122apk\dicompile1122apk\AA4_fourth_batch\thug.life.photo.sticker.maker-588
['/thug/life/photo/sticker/maker/']
Total hits: 1194
Unique URLs: 105
URLs with smali usage: 58
By source: Counter({'res': 1076, 'smali': 79, 'unknown': 28, 'assets': 9, 'manifest': 2})
Privacy-like candidates: 2
- https://firebase.google.com/support/privacy/init-options
- https://sites.google.com/flocmedia.com/thuglifepiceditorprivacypolicy/home
Wrote JSON: 1122apk\1122apk_privacy_policy_url\AA4_fourth_batch\thug.life.photo.sticker.maker-588_urls.json
Wrote CSV : 1122apk\1122apk_privacy_policy_url\AA4_fourth_batch\thug.life.photo.sticker.maker-588_urls.csv
Candidates: 2
Kept      : 2
Dropped   : 0

--- KEPT (top 30) ---
- https://firebase.google.com/support/privacy/init-options
   warn: third-party hosting domain (weak signal)
- https://sites.google.com/flocmedia.com/thuglifepiceditorprivacypolicy/home
   warn: third-party hosting domain (weak signal)

--- DROPPED (top 30) ---
Wrote: 1122apk\1122apk_privacy_po

Iterate through each `apkname-versioncode_privacy_candidates_kept.json` file, extract the PP URLs where the "keep_reasons" is "referenced from app package code (smali path)", and finally save the result as a JSON object in the format: `apkname-versioncode: [pp urls]`.

In [ ]:
# 1. Set the base path: OUT_DIR = Path(r"1122apk/1122apk_privacy_policy_url")
base_path = Path(r"two_examples/1122apk_privacy_policy_url") 
output_file = Path(rf"{base_path}/out/extracted_pp_urls.json")
# print(output_file)
# output_file.mkdir(parents=True, exist_ok=True)

results = {}

# 2. Iterate through all files in the directory ending with `_privacy_candidates_kept.json`.
for json_file in base_path.glob("*_privacy_candidates_kept.json"):
    
    # Extract the key (apkname-versioncode) by removing the "_privacy_candidates_kept.json" suffix.
    app_key = json_file.name.replace("_privacy_candidates_kept.json", "")
    
    pp_urls = []
    
    try:
        with open(json_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
            # Assuming the data is in list format, iterate through each item.
            for item in data:
                # Check if `keep_reasons` matches the target string.
                # Note: `in` or `==` is used here, depending on the actual JSON format (which could be a list or a string).
                reasons = item.get("keep_reasons", [])
                
                # Compatibility handling: If `keep_reasons` is a list, check for existence; if it is a string, perform a direct comparison.
                target_reason = "referenced from app package code (smali path)"
                if (isinstance(reasons, list) and target_reason in reasons) or (reasons == target_reason):
                    url = item.get("url")
                    if url:
                        pp_urls.append(url)
        
        # Only store the results if the URL meets the criteria.
        if pp_urls:
            results[app_key] = pp_urls
            
    except Exception as e:
        print(f"处理文件 {json_file.name} 时出错: {e}")

# 3. Save the results as a new JSON file.

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

# print(f"Extraction complete! Results saved to: {output_file}")
print(f"Processed valid data for a total of {len(results)} applications.")

Check if that PP existed in the Wayback Machine during that past period—this code needs to be added.

Iterate through the saved document (`found_privacy_policy_url.txt`) and the listed privacy policy URLs. Search the Wayback Machine for the history of each URL to check for records from October 2024 and the same month one year prior.

Iterate through the previously saved file `1122apk/1122apk_privacy_policy_url/extracted_pp_urls.json` (where keys are APK names and values ​​are the corresponding privacy policy URLs). For each APK's URL, search the Wayback Machine for its history to check for records within a one-year window around October 2024.

Iterate through `extracted_pp_urls.json`
2️⃣ Query the Wayback Machine
3️⃣ Locate the snapshot closest to 2024-10-15
4️⃣ Write the result to a CSV file (intermediate result)
5️⃣ If a snapshot exists → download the TXT version instead of the HTML

In [8]:
import os
import json
import csv
import time
import random
import requests
from urllib.parse import urlparse
import re
from html import unescape

In [ ]:
INPUT_JSON = r"1122apk/two_examples/1122apk_privacy_policy_url/out/extracted_pp_urls.json" #r"1122apk/1122apk_privacy_policy_url/out/extracted_pp_urls.json"
OUTPUT_DIR = r"1122apk/two_examples/1122apk_privacy_policy_url/pp_txt_2024_10" #r"1122apk/1122apk_privacy_policy_url/pp_html_2024_10"
CSV_FILE = r"1122apk/two_examples/1122apk_privacy_policy_url/out/pp_wayback_result.csv" #"1122apk/1122apk_privacy_policy_url/out/pp_wayback_result.csv"

# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
FROM_TS = "20231001"
TO_TS   = "20251031"
TARGET_TS = "20241015000000"   # Used to find the record closest to 2024-10-15.
CDX_API = "https://web.archive.org/cdx/search/cdx"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0"
})

In [ ]:
def normalize_url(url):
    if not url:
        return None
    url = url.strip()
    if not url.startswith(("http://", "https://")):
        url = "http://" + url
    return url


def ts_distance(a, b):
    return abs(int(a) - int(b))


def find_closest_snapshot(url):

    params = {
        "url": url,
        "from": FROM_TS,
        "to": TO_TS,
        "output": "json",
        "fl": "timestamp,original",
        "collapse": "digest",
        "filter": "statuscode:200"
    }

    r = session.get(CDX_API, params=params, timeout=30)

    if r.status_code != 200:
        return None

    data = r.json()

    if len(data) <= 1:
        return None

    header = data[0]
    rows = data[1:]

    records = []

    for row in rows:
        item = dict(zip(header, row))
        records.append(item)

    closest = min(records, key=lambda x: ts_distance(x["timestamp"], TARGET_TS))

    return closest



def html_to_text(html):
    # Remove script/style blocks.
    html = re.sub(r"(?is)<script.*?>.*?</script>", " ", html)
    html = re.sub(r"(?is)<style.*?>.*?</style>", " ", html)

    # Add line breaks after common block tags to avoid concatenating all text.
    html = re.sub(r"(?i)</p>|<br\s*/?>|</div>|</li>|</tr>|</h[1-6]>", "\n", html)
    html = re.sub(r"(?i)<li[^>]*>", "- ", html)

    # Remove all tags.
    text = re.sub(r"(?s)<[^>]+>", " ", html)

    # Unescape HTML entities.
    text = unescape(text)

    # Clean up extra whitespace.
    text = re.sub(r"\r\n|\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n\s*\n+", "\n\n", text)

    return text.strip()


def download_snapshot_1(apk_name, snapshot):

    ts = snapshot["timestamp"]
    url = snapshot["original"]

    wayback_url = f"https://web.archive.org/web/{ts}/{url}"

    out_file = os.path.join(OUTPUT_DIR, apk_name + ".html")

    try:
        r = session.get(wayback_url, timeout=60)

        if r.status_code != 200:
            return False

        with open(out_file, "wb") as f:
            f.write(r.content)

        return True

    except Exception:
        return False

def download_snapshot(apk_name, snapshot):

    ts = snapshot["timestamp"]
    url = snapshot["original"]

    wayback_url = f"https://web.archive.org/web/{ts}/{url}"

    out_file = os.path.join(OUTPUT_DIR, apk_name + ".txt")

    try:
        r = session.get(wayback_url, timeout=60)

        if r.status_code != 200:
            return False

        # Decode using the encoding detected by requests when possible.
        r.encoding = r.apparent_encoding or r.encoding
        text = html_to_text(r.text)

        with open(out_file, "w", encoding="utf-8") as f:
            f.write(text)

        return True

    except Exception:
        return False


def append_csv(row):

    file_exists = os.path.exists(CSV_FILE)

    with open(CSV_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "apk_name",
                "pp_url",
                "timestamp",
                "wayback_url",
                "has_snapshot",
                "downloaded"
            ]
        )

        if not file_exists:
            writer.writeheader()

        writer.writerow(row)

In [16]:
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

total = len(data)

print("Total:", total)

for i, (apk_name, urls) in enumerate(data.items(), 1):

    if not urls:
        url = None
    else:
        url = urls[0]


    url = normalize_url(url)

    print(f"[{i}/{total}] {apk_name}")

    if not url:

        append_csv({
            "apk_name": apk_name,
            "pp_url": "",
            "timestamp": "",
            "wayback_url": "",
            "has_snapshot": False,
            "downloaded": False
        })

        continue

    snapshot = find_closest_snapshot(url)

    if not snapshot:

        print("   no snapshot")

        append_csv({
            "apk_name": apk_name,
            "pp_url": url,
            "timestamp": "",
            "wayback_url": "",
            "has_snapshot": False,
            "downloaded": False
        })

        continue

    ts = snapshot["timestamp"]
    wayback_url = f"https://web.archive.org/web/{ts}/{url}"

    print("   snapshot:", ts)

    downloaded = download_snapshot(apk_name, snapshot)

    append_csv({
        "apk_name": apk_name,
        "pp_url": url,
        "timestamp": ts,
        "wayback_url": wayback_url,
        "has_snapshot": True,
        "downloaded": downloaded
    })

    time.sleep(1)

print("Done.")

Total: 4
[1/4] com.jvstudios.gpstracker-254
   snapshot: 20241224121427
[2/4] com.jvstudios.gpstracker-258
   snapshot: 20241224121427
[3/4] com.locator.gpstracker.phone-153
   snapshot: 20241224133705
[4/4] com.locator.gpstracker.phone-154
   snapshot: 20241224133705
Done.
